# MSDS 455 Assignment 03: Organ Transplant Waitlist vs Transplants

This notebook prepares OPTN waitlist and transplant data for a final side-by-side treemap visualization.

## 1. Imports

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

## 2. File Paths

In [ ]:
# The notebook is expected to run from the notebooks/ directory.
# This fallback also supports running from the assignment directory.
current_dir = Path.cwd().resolve()
ASSIGNMENT_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir

RAW_DIR = ASSIGNMENT_DIR / "data" / "raw"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WAITLIST_RAW_PATH = RAW_DIR / "Organ_by_Diagnosis.csv"
TRANSPLANTS_RAW_PATH = RAW_DIR / "Transplant___Organ_by_Recipient_Diagnosis.csv"

FINAL_HTML_PATH = OUTPUT_DIR / "waitlist_vs_2025_transplants_scaled_treemaps.html"
FINAL_PNG_PATH = OUTPUT_DIR / "waitlist_vs_2025_transplants_scaled_treemaps.png"
FINAL_SVG_PATH = OUTPUT_DIR / "waitlist_vs_2025_transplants_scaled_treemaps.svg"

## 3. Load Raw Data

In [ ]:
waitlist_raw = pd.read_csv(WAITLIST_RAW_PATH)
transplants_raw = pd.read_csv(TRANSPLANTS_RAW_PATH)

print(f"Waitlist raw shape: {waitlist_raw.shape}")
print(f"Transplants raw shape: {transplants_raw.shape}")

## 4. Clean and Transform Data

In [ ]:
ORGAN_COLORS = {
    "Kidney": "#6f8f72",
    "Liver": "#7c9a92",
    "Heart": "#9c7a7a",
    "Lung": "#c2a878",
    "Pancreas": "#a8b3a1",
    "Heart / Lung": "#5f7a74",
    "Kidney / Pancreas": "#4f6f55",
    "Intestine": "#b8aa8f",
    "VCA - abdominal wall": "#d6c7aa",
    "VCA - external male genitalia": "#d6c7aa",
    "VCA - head and neck": "#d6c7aa",
    "VCA - other genitourinary organ": "#d6c7aa",
    "VCA - upper limb": "#d6c7aa",
    "VCA - uterus": "#d6c7aa",
}


def clean_organ_diagnosis_export(df, value_name):
    df = df.copy()
    df = df.rename(columns={df.columns[0]: "diagnosis"})
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")], errors="ignore")
    df["diagnosis"] = df["diagnosis"].astype(str).str.strip()

    aggregate_diagnoses = ["all diagnosis", "all diagnoses", "total", "totals", "all"]
    df = df[~df["diagnosis"].str.lower().isin(aggregate_diagnoses)].copy()

    aggregate_columns = ["all organs", "all organ", "to date", "to_date", "year", "years"]
    df = df.drop(
        columns=[c for c in df.columns if str(c).strip().lower() in aggregate_columns],
        errors="ignore",
    )

    organ_cols = [c for c in df.columns if c != "diagnosis"]
    for col in organ_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"": "0", "nan": "0", "*": "0"})
        )
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    long_df = df.melt(
        id_vars=["diagnosis"],
        value_vars=organ_cols,
        var_name="organ",
        value_name=value_name,
    )
    long_df["organ"] = long_df["organ"].astype(str).str.strip()

    return long_df[(long_df[value_name] > 0) & (long_df["organ"].str.lower() != "all organs")].copy()


wait_long = clean_organ_diagnosis_export(waitlist_raw, "waitlist_count")
tx_long = clean_organ_diagnosis_export(transplants_raw, "transplants")

## 5. Exploratory Data Analysis

In [ ]:
print("Cleaned dataset shapes")
print(f"Waitlist long shape: {wait_long.shape}")
print(f"Transplants long shape: {tx_long.shape}")

print("\nMissing values")
print("Waitlist:")
print(wait_long.isna().sum())
print("\nTransplants:")
print(tx_long.isna().sum())

In [ ]:
waitlist_totals_by_organ = (
    wait_long.groupby("organ", as_index=False)["waitlist_count"]
    .sum()
    .sort_values("waitlist_count", ascending=False)
)

transplant_totals_by_organ = (
    tx_long.groupby("organ", as_index=False)["transplants"]
    .sum()
    .sort_values("transplants", ascending=False)
)

print("Waitlist totals by organ")
print(waitlist_totals_by_organ.to_string(index=False))

print("\n2025 transplant totals by organ")
print(transplant_totals_by_organ.to_string(index=False))

In [ ]:
sanity_checks = pd.DataFrame({
    "check": [
        "All Diagnosis rows remaining in waitlist",
        "All Diagnosis rows remaining in transplants",
        "All Organs rows remaining in waitlist",
        "All Organs rows remaining in transplants",
    ],
    "count": [
        wait_long["diagnosis"].str.lower().isin(["all diagnosis", "all diagnoses"]).sum(),
        tx_long["diagnosis"].str.lower().isin(["all diagnosis", "all diagnoses"]).sum(),
        (wait_long["organ"].str.lower() == "all organs").sum(),
        (tx_long["organ"].str.lower() == "all organs").sum(),
    ],
})

print(sanity_checks.to_string(index=False))

### EDA Summary

Aggregate rows and columns such as All Diagnosis and All Organs were removed to avoid double-counting. The data was reshaped from wide to long format to support hierarchical visualization. Kidney demand dominates the waitlist, driven largely by chronic conditions such as diabetes and hypertension. 2025 transplants follow similar patterns but at a smaller scale.

## 6. Final Visualization

In [ ]:
wait_total = wait_long["waitlist_count"].sum()
tx_total = tx_long["transplants"].sum()

fig_wait = px.treemap(
    wait_long,
    path=["organ", "diagnosis"],
    values="waitlist_count",
    color="organ",
    color_discrete_map=ORGAN_COLORS,
)

fig_tx = px.treemap(
    tx_long,
    path=["organ", "diagnosis"],
    values="transplants",
    color="organ",
    color_discrete_map=ORGAN_COLORS,
)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "domain"}, {"type": "domain"}]],
    subplot_titles=(
        f"Current Waitlist Demand<br>{wait_total:,.0f} registrations",
        f"2025 Transplants<br>{tx_total:,.0f} transplants",
    ),
    column_widths=[
        wait_total / (wait_total + tx_total),
        tx_total / (wait_total + tx_total),
    ],
    horizontal_spacing=0.04,
)

for trace in fig_wait.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig_tx.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_traces(textinfo="label+value+percent entry")
fig.update_traces(hovertemplate="<b>%{label}</b><br>%{value:,.0f}<extra></extra>")

fig.update_layout(
    title=dict(
        text="Organ Transplant Waitlist vs Transplants by Organ and Diagnosis",
        x=0.5,
        xanchor="center",
        font=dict(size=19),
    ),
    margin=dict(t=90, l=10, r=10, b=160),
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    font=dict(color="#2f3e34"),
)

for annotation in fig.layout.annotations[:2]:
    annotation.update(font=dict(size=13), y=0.98)

fig.add_annotation(
    x=0.5,
    y=-0.12,
    xref="paper",
    yref="paper",
    text=(
        "• Kidney demand dominates the waitlist, driven largely by chronic conditions like diabetes and hypertension"
        "<br>"
        "• Transplants follow similar patterns but at a much smaller scale, highlighting a supply constraint"
    ),
    showarrow=False,
    font=dict(size=12),
    align="center",
)

fig.add_annotation(
    x=0.5,
    y=-0.20,
    xref="paper",
    yref="paper",
    text="Source: OPTN National Data — Waitlist & Transplants",
    showarrow=False,
    font=dict(size=10),
    align="center",
)

fig.show()

## 7. Export Outputs

In [ ]:
fig.write_html(FINAL_HTML_PATH)

try:
    fig.write_image(FINAL_PNG_PATH, scale=2)
    fig.update_layout(paper_bgcolor="#ffffff", plot_bgcolor="#ffffff")
    fig.write_image(FINAL_SVG_PATH)
    print(f"Saved HTML: {FINAL_HTML_PATH}")
    print(f"Saved PNG: {FINAL_PNG_PATH}")
    print(f"Saved SVG: {FINAL_SVG_PATH}")
except Exception as e:
    print("HTML saved. Static image export failed:")
    print(e)